In [ ]:
import os
import pandas as pd


ROOT_PATH = "/content/drive/MyDrive/mashob/labaKredit"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Google Drive уже смонтирован или монтирование не требуется.")


train_file_path = os.path.join(ROOT_PATH, 'train.csv')
test_file_path = os.path.join(ROOT_PATH, 'test.csv')

Mounted at /content/drive


In [ ]:
train_df = pd.read_csv(train_file_path)
test_df = pd.read_csv(test_file_path)

In [ ]:
print("Число пропусков в тренировочном:", train_df.isnull().sum().sum())

print("\nЧисло пропусков в тесте:", test_df.isnull().sum().sum())

Число пропусков в тренировочном: 0

Число пропусков в тесте: 0


In [ ]:


def predict_credit(row):
    credit_score = row['credit_score']
    dti = row['debt_to_income_ratio']
    employment = row['employment_status']
    income = row['annual_income']
    loan = row['loan_amount']

    if (credit_score >= 750 and
        dti <= 0.25 and
        employment == 'Employed' and
        income >= 25000 and loan <= 30000):
        return 1
    else:
        return 0


test_df['prediction'] = test_df.apply(predict_credit, axis=1)

result_df = test_df[['id', 'prediction']].copy()
result_df.to_csv('credit_predictions.csv', index=False)

print(" Предсказания сохранены в файл 'credit_predictions.csv'")


 Предсказания сохранены в файл 'credit_predictions.csv'


In [ ]:
print(result_df.head(20))

        id  prediction
0   593994           0
1   593995           1
2   593996           0
3   593997           1
4   593998           1
5   593999           1
6   594000           1
7   594001           1
8   594002           1
9   594003           0
10  594004           0
11  594005           1
12  594006           0
13  594007           0
14  594008           1
15  594009           0
16  594010           1
17  594011           1
18  594012           0
19  594013           1


In [ ]:
count_ones = result_df['prediction'].sum()

print(f"Количество предсказаний со значением 1: {count_ones}")

Количество предсказаний со значением 1: 15228


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score

import xgboost as xgb


TARGET = 'loan_paid_back'
ID_COL = 'id'


FEATURES = ['credit_score', 'debt_to_income_ratio', 'employment_status', 'annual_income', 'loan_amount']


X = train_df[FEATURES].copy()
y = train_df[TARGET]

categorical_cols = ['employment_status']
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()

    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le


X_test = test_df[FEATURES].copy()

for col in categorical_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(str)

        X_test[col] = X_test[col].map(
            lambda x: le_dict[col].transform([x])[0] if x in le_dict[col].classes_ else -1
        )


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


#  Random Forest
print("Обучение Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


rf_pred_val = rf_model.predict(X_val)
rf_pred_proba_val = rf_model.predict_proba(X_val)[:, 1]
rf_acc = accuracy_score(y_val, rf_pred_val)
rf_auc = roc_auc_score(y_val, rf_pred_proba_val)
print(f"Random Forest — Accuracy: {rf_acc:.4f}, ROC AUC: {rf_auc:.4f}")


rf_test_pred = rf_model.predict(X_test)

rf_result = pd.DataFrame({
    'id': test_df[ID_COL],
    'prediction': rf_test_pred
})
rf_path = os.path.join(ROOT_PATH, 'rf_predictions.csv')
rf_result.to_csv(rf_path, index=False)
print(f"Random Forest предсказания сохранены: {rf_path}")

# XGBoost
print("\nОбучение XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)


xgb_pred_val = xgb_model.predict(X_val)
xgb_pred_proba_val = xgb_model.predict_proba(X_val)[:, 1]
xgb_acc = accuracy_score(y_val, xgb_pred_val)
xgb_auc = roc_auc_score(y_val, xgb_pred_proba_val)
print(f"XGBoost — Accuracy: {xgb_acc:.4f}, ROC AUC: {xgb_auc:.4f}")


xgb_test_pred = xgb_model.predict(X_test)


xgb_result = pd.DataFrame({
    'id': test_df[ID_COL],
    'prediction': xgb_test_pred
})
xgb_path = os.path.join(ROOT_PATH, 'xgb_predictions.csv')
xgb_result.to_csv(xgb_path, index=False)
print(f"XGBoost предсказания сохранены: {xgb_path}")


metrics_path = os.path.join(ROOT_PATH, 'model_comparison_metrics.txt')
with open(metrics_path, 'w') as f:
    f.write("Сравнение моделей на валидационной выборке:\n\n")
    f.write(f"Random Forest:\n  Accuracy = {rf_acc:.4f}\n  ROC AUC = {rf_auc:.4f}\n\n")
    f.write(f"XGBoost:\n  Accuracy = {xgb_acc:.4f}\n  ROC AUC = {xgb_auc:.4f}\n")
print(f"\nМетрики сохранены: {metrics_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Обучение Random Forest...
Random Forest — Accuracy: 0.9012, ROC AUC: 0.9050
Random Forest предсказания сохранены: /content/drive/MyDrive/mashob/labaKredit/rf_predictions.csv

Обучение XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [05:32:11] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost — Accuracy: 0.9047, ROC AUC: 0.9197
XGBoost предсказания сохранены: /content/drive/MyDrive/mashob/labaKredit/xgb_predictions.csv

Метрики сохранены: /content/drive/MyDrive/mashob/labaKredit/model_comparison_metrics.txt
